# 1. 패키지 설치

In [1]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 2. Knowledge Base 구성을 위한 데이터 생성

- [RecursiveCharacterTextSplitter](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/)를 활용한 데이터 chunking
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
    - LangChain에서 다양한 [TextSplitter](https://python.langchain.com/v0.2/docs/how_to/#text-splitters)들을 제공
- `chunk_size` 는 split 된 chunk의 최대 크기
- `chunk_overlap`은 앞 뒤로 나뉘어진 chunk들이 얼마나 겹쳐도 되는지 지정

In [ ]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

/var/folders/jw/qym3n0892n51wr3xcmd925d00000gn/T/ipykernel_74692/437428471.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [ ]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터를 처음 저장할 때 
index_name = 'tax-upstage-index'
database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)
retrieved_docs = database.similarity_search(query, k=3)

/Users/nsj/.pyenv/versions/inflearn-llm-application2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 3. 답변 생성을 위한 Retrieval

- `Chroma`에 저장한 데이터를 유사도 검색(`similarity_search()`)를 활용해서 가져옴

In [ ]:
query = '근로소득자 9천만원 세금 어케됨?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정


In [6]:
retrieved_docs

[Document(id='9775036e-f854-4b72-8815-4ea317549644', metadata={'source': './tax.docx'}, page_content='제5관 근로소득공제ㆍ연금소득공제 및 퇴직소득공제 <개정 2009. 12. 31.>\n\n\n\n제47조(근로소득공제) ①근로소득이 있는 거주자에 대해서는 해당 과세기간에 받는 총급여액에서 다음의 금액을 공제한다. 다만, 공제액이 2천만원을 초과하는 경우에는 2천만원을 공제한다. <개정 2012. 1. 1., 2014. 1. 1., 2019. 12. 31.>\n\n\n\n② 일용근로자에 대한 공제액은 제1항에도 불구하고 1일 15만원으로 한다.<개정 2018. 12. 31.>\n\n③ 근로소득이 있는 거주자의 해당 과세기간의 총급여액이 제1항 또는 제2항의 공제액에 미달하는 경우에는 그 총급여액을 공제액으로 한다.\n\n④ 제1항부터 제3항까지의 규정에 따른 공제를 “근로소득공제”라 한다.\n\n⑤ 제1항의 경우에 2인 이상으로부터 근로소득을 받는 사람(일용근로자는 제외한다)에 대하여는 그 근로소득의 합계액을 총급여액으로 하여 제1항에 따라 계산한 근로소득공제액을 총급여액에서 공제한다.<개정 2010. 12. 27.>\n\n⑥ 삭제<2010. 12. 27.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제47조의2(연금소득공제) ①연금소득이 있는 거주자에 대해서는 해당 과세기간에 받은 총연금액(분리과세연금소득은 제외하며, 이하 이 항에서 같다)에서 다음 표에 규정된 금액을 공제한다. 다만, 공제액이 900만원을 초과하는 경우에는 900만원을 공제한다. <개정 2013. 1. 1.>\n\n\n\n② 제1항에 따른 공제를 “연금소득공제”라 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제48조(퇴직소득공제) ① 퇴직소득이 있는 거주자에 대해서는 해당 과세기간의 퇴직소득금액에서 제1호의 구분에 따른 금액을 공제하고, 그 금액을 근속연수(1년 미만의 기간이 있는 

# 4. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [ ]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_upstage import ChatUpstage


# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터를 처음 저장할 때 
index_name = 'tax-upstage-index'
database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)
retrieved_docs = database.similarity_search(query, k=3)
llm = ChatUpstage()

In [ ]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터를 처음 저장할 때 
index_name = 'tax-upstage-index'
database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)
retrieved_docs = database.similarity_search(query, k=3)
llm = ChatUpstage()

prompt = ChatPromptTemplate.from_messages([
    ("human",
     "You are an assistant for question-answering tasks. "
     "Use the following retrieved context to answer the question. "
     "If you don't know the answer, just say that you don't know. "
     "Use three sentences maximum and keep the answer concise.\n\n"
     "Question: {question}\nContext: {context}\nAnswer:")
])

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain({"query": query})

# 5. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [9]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [10]:
ai_message = qa_chain({"query": query})

/var/folders/jw/qym3n0892n51wr3xcmd925d00000gn/T/ipykernel_74692/3455095564.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ai_message = qa_chain({"query": query})


In [13]:
# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장
ai_message = qa_chain.invoke({"query": query})



In [14]:
ai_message

{'query': '근로소득자 9천만원 세금 어케됨?',
 'result': '근로소득세 계산은 총급여액에서 근로소득공제액을 차감하여 근로소득금액을 산출한 후, 세율을 적용하여 계산합니다. 총급여액이 9천만원일 경우, 근로소득공제액은 2천만원으로 적용됩니다. 따라서 근로소득금액은 7천만원이 되며, 이에 대한 세금을 계산해야 합니다. 소득세율은 과세표준에 따라 달라지므로 정확한 세금을 알기 위해서는 추가적인 정보가 필요합니다.'}

In [ ]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터를 처음 저장할 때 
index_name = 'tax-upstage-index'
database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)
retrieved_docs = database.similarity_search(query, k=3)
llm = ChatUpstage()

prompt = ChatPromptTemplate.from_messages([
    ("human",
     "You are an assistant for question-answering tasks. "
     "Use the following retrieved context to answer the question. "
     "If you don't know the answer, just say that you don't know. "
     "Use three sentences maximum and keep the answer concise.\n\n"
     "Question: {question}\nContext: {context}\nAnswer:")
])

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain({"query": query})

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain

In [16]:
new_question = dictionary_chain.invoke({"question": query})

In [17]:
new_question

'근로소득자 9천만원 세금 어떻게 되나요?'

In [18]:
ai_response = tax_chain.invoke({"question": query})
ai_response


{'query': '근로소득자인 거주자 9천만원 세금 어케됨?',
 'result': '근로소득자인 거주자의 총급여액이 9천만원일 경우, 근로소득공제를 적용하면 과세표준이 됩니다. 근로소득공제는 총급여액에서 다음의 금액을 공제하는데, 공제액이 2천만원을 초과하는 경우에는 2천만원을 공제합니다. 따라서, 총급여액이 9천만원이면 2천만원을 공제하고, 7천만원이 과세표준이 됩니다. 이후, 종합소득공제를 적용하여 최종적으로 납부할 세금을 계산하게 됩니다.'}